In [ ]:
import json
from pathlib import Path
from typing import Any

# Directory containing the evaluation results
data_dir = Path("/home/igarc/FactChecking-v2/evaluation_results/gemini_rag")


def extract_conversation_history(file_path: Path) -> list[dict[str, Any]]:
    """
    Extract all conversation history entries from a JSON file.

    Args:
        file_path: Path to the JSON file

    Returns:
        List of dictionaries containing conversation data and metadata
    """
    conversations = []

    with open(file_path, encoding="utf-8") as f:
        data = json.load(f)

    # Get the logs section
    logs = data.get("logs", {})

    # Iterate through all agents
    for agent_name, agent_data in logs.items():
        if agent_name == "summary":  # Skip summary section
            continue

        history = agent_data.get("history", [])

        # Check each history entry
        for hist_entry in history:
            history_entry = hist_entry.get("history_entry", {})

            # Filter for conversation type
            if history_entry.get("type") == "conversation":
                conversations.append(
                    {
                        "file": file_path.name,
                        "agent": agent_name,
                        "date": hist_entry.get("date"),
                        "run_time": hist_entry.get("run_time"),
                        "cost": hist_entry.get("cost"),
                        "data": history_entry.get("data"),
                    }
                )

    return conversations


# Load all JSON files and extract conversations
all_conversations = []

# Get all JSON files in the directory
json_files = sorted(data_dir.glob("*.json"))

print(f"Found {len(json_files)} JSON files")

for json_file in json_files:
    try:
        conversations = extract_conversation_history(json_file)
        all_conversations.extend(conversations)
        print(f"Processed {json_file.name}: {len(conversations)} conversation entries")
    except Exception as e:
        print(f"Error processing {json_file.name}: {e}")

print(f"\nTotal conversation entries extracted: {len(all_conversations)}")

# Display a sample conversation
if all_conversations:
    print("\nSample conversation entry:")
    sample = all_conversations[0]
    print(f"File: {sample['file']}")
    print(f"Agent: {sample['agent']}")
    print(f"Date: {sample['date']}")
    print(f"Number of messages: {len(sample['data'])}")

Found 303 JSON files
Processed c0000.json: 3 conversation entries
Processed c0001.json: 3 conversation entries
Processed c0002.json: 3 conversation entries
Processed c0003.json: 3 conversation entries
Processed c0004.json: 3 conversation entries
Processed c0005.json: 3 conversation entries
Processed c0006.json: 3 conversation entries
Processed c0007.json: 3 conversation entries
Processed c0008.json: 3 conversation entries
Processed c0009.json: 3 conversation entries
Processed c0010.json: 3 conversation entries
Processed c0011.json: 3 conversation entries
Processed c0012.json: 3 conversation entries
Processed c0013.json: 3 conversation entries
Processed c0014.json: 3 conversation entries
Processed c0015.json: 3 conversation entries
Processed c0016.json: 3 conversation entries
Processed c0017.json: 3 conversation entries
Processed c0018.json: 3 conversation entries
Processed c0019.json: 3 conversation entries
Processed c0020.json: 3 conversation entries
Processed c0021.json: 3 conversati

In [ ]:
# Group conversations by agent type
from collections import defaultdict

conversations_by_agent = defaultdict(list)
for conv in all_conversations:
    conversations_by_agent[conv["agent"]].append(conv)

print("Conversations by agent:")
for agent, convs in conversations_by_agent.items():
    print(f"  {agent}: {len(convs)} conversations")

Conversations by agent:
  GenSearchesAgent: 300 conversations
  ArticleWriterAgent: 300 conversations
  MetadataAgent: 300 conversations


In [ ]:
dataset = []

chat_2_topic = {
    "c": "Ciencia",
    "i": "Igualdad",
    "p": "Política",
}
for conv in all_conversations:
    data = {
        "id": conv["file"].split(".")[0],
        "messages": conv["data"],
        "agent": conv["agent"],
        "date": conv["date"],
        "topic": chat_2_topic[conv["file"][0]],
    }
    dataset.append(data)

print(len(dataset))
dataset[0]

900


{'id': 'c0000',
 'messages': [{'role': 'system',
   'content': 'You are a search query strategist specialized in fact-checking research. Your task is to generate targeted Google search queries that will efficiently retrieve the information needed to answer critical research questions about a statement.\n\n**Your Mission:**\nYou will receive:\n1. A statement that needs fact-checking\n2. A set of critical research questions designed to verify or challenge the statement\n\nYour goal is to create 3-5 strategic Google search queries that will gather the most relevant information to answer these questions comprehensively.\n\n**Search Strategy Guidelines:**\n\n**1. Optimize for Information Retrieval:**\n- Create searches that target primary sources, official data, and authoritative content\n- Use specific keywords that are likely to return high-quality, credible results\n- Target specific organizations, institutions, or authoritative sources when relevant\n\n**2. Search Query Best Practices:*

In [11]:
from datasets import Dataset

dataset = Dataset.from_list(dataset)
dataset[0]

dataset.push_to_hub("Iker/factchecking-calibration-dataset")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/Iker/factchecking-calibration-dataset/commit/4f66e900a75f2fb7fee8eb621a1fff0e44411f03', commit_message='Upload dataset', commit_description='', oid='4f66e900a75f2fb7fee8eb621a1fff0e44411f03', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Iker/factchecking-calibration-dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Iker/factchecking-calibration-dataset'), pr_revision=None, pr_num=None)

In [12]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
smoltalk2 = load_dataset("HuggingFaceTB/smoltalk2", "SFT")

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

SFT/LongAlign_64k_Qwen3_32B_yarn_131k_th(…):   0%|          | 0.00/135M [00:00<?, ?B/s]

...

Generating smoltalk_smollm3_systemchats_30k_no_think split:   0%|          | 0/33997 [00:00<?, ? examples/s]

Generating table_gpt_no_think split:   0%|          | 0/13203 [00:00<?, ? examples/s]

Generating tulu_3_sft_personas_instruction_following_no_think split:   0%|          | 0/29970 [00:00<?, ? exam…

Generating xlam_traces_no_think split:   0%|          | 0/59962 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/105 [00:00<?, ?it/s]

In [26]:
smoltalk2_shuffled = smoltalk2["OpenThoughts3_1.2M_think"].shuffle(seed=42)
smoltalk2_selected = smoltalk2_shuffled.select(range(4096))

In [27]:
smoltalk2_selected

Dataset({
    features: ['messages', 'chat_template_kwargs', 'source'],
    num_rows: 4096
})

In [28]:
import re

combined_dataset = []


def remove_thinking(messages):
    """Remove <think>...</think> blocks. If any opening <think> is not closed, return None to discard the example."""
    # Discard example if any message has an opening <think> without a closing </think>
    for message in messages:
        content = message.get("content", "")
        if re.search(r"<\s*think\s*>", content, flags=re.IGNORECASE) and not re.search(
            r"</\s*think\s*>", content, flags=re.IGNORECASE
        ):
            return None

    cleaned_messages = []
    for message in messages:
        cleaned_message = {"role": message["role"]}
        if "content" in message:
            cleaned_content = re.sub(
                r"<\s*think\s*>[\s\S]*?</\s*think\s*>", "", message["content"], flags=re.IGNORECASE
            )
            cleaned_message["content"] = cleaned_content.strip()
        for key in message:
            if key not in ["role", "content"]:
                cleaned_message[key] = message[key]
        cleaned_messages.append(cleaned_message)
    return cleaned_messages


for example in dataset:
    combined_dataset.append(
        {"messages": example["messages"], "source": f"{example['topic']} - {example['id']}"}
    )

for example in smoltalk2_selected:
    cleaned = remove_thinking(example["messages"])
    if cleaned is None:
        continue  # discard examples with unterminated <think>
    combined_dataset.append({"messages": cleaned, "source": f"{example['source']}"})

combined_dataset = Dataset.from_list(combined_dataset)
print(combined_dataset)
combined_dataset[-1]

Dataset({
    features: ['messages', 'source'],
    num_rows: 2465
})


{'messages': [{'content': 'A positive integer is said to be a $\\textit{mirrored number}$ if it is divisible by each of its distinct proper factors. How many numbers less than 100 are mirrored numbers?',
   'role': 'user'},
  {'content': 'To determine how many numbers less than 100 are mirrored numbers, we need to clarify the definition: a mirrored number is a positive integer divisible by each of its distinct proper factors. After considering various interpretations, we found that the correct interpretation is that a mirrored number has exactly four divisors (including the number itself), meaning it is either the product of two distinct primes (semiprimes) or the cube of a prime (prime cubes). \n\n### Key Steps:\n\n1. **Prime Cubes**: \n   - Prime cubes less than 100 are:\n     - \\(2^3 = 8\\)\n     - \\(3^3 = 27\\)\n   - Total: 2 numbers.\n\n2. **Semiprimes (Products of Two Distinct Primes)**:\n   - List semiprimes systematically:\n     - Starting with 2: \\(6, 10, 14, 22, 26, 34, 38

In [30]:
combined_dataset[-45]

{'messages': [{'content': "Tom has a total of 72 dollars in his wallet. He only has five dollar bills and ten dollar bills in his wallet. If there are a total of 16 bills in Tom's wallet, how many five dollar bills does he have if he knows that he received exactly twice as many ten dollar bills as five dollar bills from his grandmother?",
   'role': 'user'},
  {'content': "Tom has a total of 72 dollars in his wallet, consisting only of five dollar bills and ten dollar bills, with a total of 16 bills. He also received exactly twice as many ten dollar bills as five dollar bills from his grandmother. \n\nLet \\( F \\) be the number of five dollar bills and \\( T \\) be the number of ten dollar bills. The problem gives us the following equations:\n1. \\( F + T = 16 \\)\n2. \\( 5F + 10T = 72 \\)\n3. The number of ten dollar bills received from the grandmother (\\( T_g \\)) is twice the number of five dollar bills received from the grandmother (\\( F_g \\)), \\( T_g = 2F_g \\).\n\nAssuming a

In [25]:
combined_dataset

Dataset({
    features: ['messages', 'source'],
    num_rows: 1687
})

In [31]:
combined_dataset.push_to_hub("Iker/calibration-dataset")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/Iker/calibration-dataset/commit/18ec46b7b429825f4704cfd6aa69439e23d428ec', commit_message='Upload dataset', commit_description='', oid='18ec46b7b429825f4704cfd6aa69439e23d428ec', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Iker/calibration-dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Iker/calibration-dataset'), pr_revision=None, pr_num=None)